# Final Logistic Matrix Factorization Experiments

It reproduces the complete Logistic Matrix Factorization workflow:

1. 30-epoch full-capacity sweep;
2. training-budget diagnostic at \(lr=0.01\);
3. learning-rate stabilization diagnostic at \(lr=0.003\);
4. final stabilized full-capacity sweep;
5. figures, summary tables, and downloadable outputs.

## 1. Input and Output Paths

In [ ]:
from pathlib import Path

KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")

input_matches = list(KAGGLE_INPUT_DIR.rglob("interactions_real_long.csv"))

if len(input_matches) == 0:
    raise FileNotFoundError(
        "interactions_real_long.csv was not found under /kaggle/input. "
        "Attach the Kaggle dataset containing this file."
    )

if len(input_matches) > 1:
    raise RuntimeError(
        "More than one interactions_real_long.csv file was found:\n"
        + "\n".join(str(path) for path in input_matches)
        + "\nKeep only one copy attached to the notebook."
    )

INPUT_PATH = input_matches[0]

OUTPUT_DIR = KAGGLE_WORKING_DIR / "logistic_mf_final_matched_results"
RESULTS_DIR = OUTPUT_DIR / "results"
FIGURES_DIR = OUTPUT_DIR / "figures"
SPLITS_DIR = OUTPUT_DIR / "splits"
SUMMARY_DIR = OUTPUT_DIR / "summaries"
METADATA_DIR = OUTPUT_DIR / "metadata"

for directory in [
    OUTPUT_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    SPLITS_DIR,
    SUMMARY_DIR,
    METADATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Input file:", INPUT_PATH)
print("Output directory:", OUTPUT_DIR)

## 2. Imports and Reproducibility


In [ ]:
import gc
import hashlib
import json
import os
import random
import time
import warnings
import zipfile
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import FileLink, display
from torch.utils.data import DataLoader, TensorDataset

SEED = 20060709
SEED_NEG = 0
SEED_EVAL = 1


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Enable a Kaggle GPU accelerator before "
        "running the experiments."
    )

device = torch.device("cuda")

print("Python/PyTorch environment ready")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Training seed:", SEED)
print("Negative-sampling seed:", SEED_NEG)
print("Evaluation seed:", SEED_EVAL)

## 3. Experiment Configuration

The original sweep uses the initial optimization setup. The two high-\(k\)
diagnostics use the same latent dimensions and training budgets so that the
learning-rate comparison is controlled. The final sweep uses the stabilized
configuration reported in the project.

In [ ]:
NEG_RATIO_TRAIN = 4
EVAL_NEGATIVE_RATIOS = (4, 100)
WEIGHT_DECAY = 1e-6

MIN_INTERACTIONS_GRID = [10, 20, 25]

INITIAL_K_GRID = [
    1, 2, 4, 8, 16, 24, 32, 48, 64, 96, 128, 192, 256,
    384, 512, 768, 1024, 1536, 2048, 3072, 4000
]
INITIAL_EPOCHS = [30]
INITIAL_LEARNING_RATE = 1e-2
INITIAL_GRAD_CLIP = None

HIGH_K_GRID = [48, 64, 128, 256, 512, 1024, 2048]
DIAGNOSTIC_EPOCH_GRID = [30, 50, 100, 200]
DIAGNOSTIC_MIN_GRID = [10]
DIAGNOSTIC_BATCH_SIZE = 1024

TRAINING_BUDGET_LEARNING_RATE = 1e-2
TRAINING_BUDGET_GRAD_CLIP = None

STABILITY_LEARNING_RATE = 3e-3
# No clipping here: this diagnostic isolates the learning-rate change.
STABILITY_DIAGNOSTIC_GRAD_CLIP = None

FINAL_STABLE_MIN_GRID = [10, 20]
FINAL_STABLE_K_GRID = [
    1, 2, 4, 8, 16, 24, 32, 48, 64, 96, 128, 192, 256,
    384, 512, 768, 1024, 1536, 2048
]
FINAL_STABLE_EPOCHS = [50]
FINAL_STABLE_LEARNING_RATE = 3e-3
FINAL_STABLE_BATCH_SIZE = 1024
FINAL_STABLE_GRAD_CLIP = 1.0

# Resume only from files created by this notebook in /kaggle/working.
# No result file is ever loaded from /kaggle/input.
RESUME_FROM_WORKING = True

# Prevent the held-out positive from being sampled as a training negative.
EXCLUDE_TEST_ITEM_FROM_TRAIN_NEGATIVES = True

RUN_INITIAL_SWEEP = True
RUN_TRAINING_BUDGET_DIAGNOSTIC = True
RUN_STABILITY_DIAGNOSTIC = True
RUN_FINAL_STABLE_SWEEP = True

INITIAL_RESULTS_PATH = RESULTS_DIR / "logistic_mf_initial_30_epoch_sweep.csv"
TRAINING_BUDGET_RESULTS_PATH = (
    RESULTS_DIR / "logistic_mf_high_k_training_budget_lr001.csv"
)
STABILITY_RESULTS_PATH = (
    RESULTS_DIR / "logistic_mf_high_k_stability_lr003.csv"
)
FINAL_STABLE_RESULTS_PATH = (
    RESULTS_DIR / "logistic_mf_final_stabilized_sweep.csv"
)

print("Initial models:", len(MIN_INTERACTIONS_GRID) * len(INITIAL_K_GRID))
print(
    "Training-budget diagnostic models:",
    len(DIAGNOSTIC_MIN_GRID)
    * len(HIGH_K_GRID)
    * len(DIAGNOSTIC_EPOCH_GRID),
)
print(
    "Learning-rate diagnostic models:",
    len(DIAGNOSTIC_MIN_GRID)
    * len(HIGH_K_GRID)
    * len(DIAGNOSTIC_EPOCH_GRID),
)
print(
    "Final stabilized models:",
    len(FINAL_STABLE_MIN_GRID) * len(FINAL_STABLE_K_GRID),
)

## 4. Load and Validate Interaction Data

The input must contain one row per positive user--item interaction. This cell
checks identifiers, missing values, duplicated pairs, and the optional binary
response column before any split is created.

In [ ]:
df = pd.read_csv(INPUT_PATH)

REQUIRED_COLUMNS = ["customer_id_anon", "work_id"]
missing_columns = [column for column in REQUIRED_COLUMNS if column not in df.columns]

if missing_columns:
    raise ValueError(
        "The input file is missing required columns: "
        + ", ".join(missing_columns)
    )

if df[REQUIRED_COLUMNS].isna().any().any():
    missing_counts = df[REQUIRED_COLUMNS].isna().sum()
    raise ValueError(
        "Missing user or item identifiers were found:\n"
        + missing_counts.to_string()
    )

duplicated_pairs = int(
    df.duplicated(["customer_id_anon", "work_id"]).sum()
)

if duplicated_pairs != 0:
    raise ValueError(
        f"The input contains {duplicated_pairs} duplicated user-item pairs. "
        "The final interaction file must already be aggregated."
    )

if "binary_utility" in df.columns:
    binary_values = set(
        pd.to_numeric(df["binary_utility"], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
    )
    if binary_values != {1}:
        raise ValueError(
            "interactions_real_long.csv must contain positive interactions only. "
            f"Found binary_utility values: {sorted(binary_values)}"
        )

n_users_raw = int(df["customer_id_anon"].nunique())
n_items_raw = int(df["work_id"].nunique())
n_interactions_raw = int(len(df))
density_raw = n_interactions_raw / (n_users_raw * n_items_raw)

raw_dataset_summary = pd.DataFrame(
    {
        "metric": [
            "Users",
            "Items",
            "Positive user-item interactions",
            "Density (%)",
            "Sparsity (%)",
        ],
        "value": [
            n_users_raw,
            n_items_raw,
            n_interactions_raw,
            100 * density_raw,
            100 * (1 - density_raw),
        ],
    }
)

raw_summary_path = SUMMARY_DIR / "raw_dataset_summary.csv"
raw_dataset_summary.to_csv(raw_summary_path, index=False)

print("Input shape:", df.shape)
display(raw_dataset_summary)
print("Saved:", raw_summary_path)

## 5. Train-Test Split and Negative Sampling

A fixed random leave-one-out split is created after each minimum-interaction
filter. Cold test items are removed. Training negatives are sampled once and
reused across all \(k\)-values. The held-out test positive is explicitly excluded
from the training-negative pool.

In [ ]:
OPTIONAL_OUTPUT_COLUMNS = [
    "customer_id_anon",
    "work_id",
    "distinct_orders",
    "units",
    "total_spend",
    "first_purchase",
    "last_purchase",
    "binary_utility",
]


def build_train_test(interactions, min_interactions):
    per_user_count = interactions.groupby(
        "customer_id_anon"
    )["work_id"].transform("size")

    kept = interactions[per_user_count >= min_interactions].copy()

    # Preserve the original project protocol: fixed global shuffle followed by
    # one held-out interaction per user.
    kept = kept.sample(frac=1, random_state=SEED).reset_index(drop=True)
    within_user_position = kept.groupby("customer_id_anon").cumcount()

    test = kept[within_user_position == 0].copy()
    train = kept[within_user_position != 0].copy()

    train_items = set(train["work_id"].unique())
    cold_mask = ~test["work_id"].isin(train_items)
    cold_removed = int(cold_mask.sum())
    test = test[~cold_mask].copy()

    if train.duplicated(["customer_id_anon", "work_id"]).any():
        raise ValueError(
            f"MIN{min_interactions}: duplicated user-item pairs in train."
        )

    if test.duplicated(["customer_id_anon", "work_id"]).any():
        raise ValueError(
            f"MIN{min_interactions}: duplicated user-item pairs in test."
        )

    leakage = test.merge(
        train[["customer_id_anon", "work_id"]],
        on=["customer_id_anon", "work_id"],
        how="inner",
    )

    if not leakage.empty:
        raise ValueError(
            f"MIN{min_interactions}: {len(leakage)} train-test duplicate pairs."
        )

    minimum_train_count = int(
        train.groupby("customer_id_anon").size().min()
    )

    if minimum_train_count < min_interactions - 1:
        raise ValueError(
            f"MIN{min_interactions}: a user has fewer than "
            f"{min_interactions - 1} training positives."
        )

    output_columns = [
        column for column in OPTIONAL_OUTPUT_COLUMNS if column in interactions.columns
    ]

    train_output = train[output_columns].sort_values(
        ["customer_id_anon", "work_id"]
    )
    test_output = test[output_columns].sort_values(
        ["customer_id_anon", "work_id"]
    )

    train_path = SPLITS_DIR / f"train_min{min_interactions}.csv"
    test_path = SPLITS_DIR / f"test_min{min_interactions}.csv"

    train_output.to_csv(train_path, index=False)
    test_output.to_csv(test_path, index=False)

    return train, test, cold_removed, len(kept), train_path, test_path


def prepare_protocol_data(
    interactions,
    min_interactions,
    neg_ratio=NEG_RATIO_TRAIN,
):
    (
        train,
        test,
        cold_removed,
        kept_interactions,
        train_path,
        test_path,
    ) = build_train_test(interactions, min_interactions)

    # Keep deterministic first-occurrence mappings to preserve the original
    # experiment construction.
    users = train["customer_id_anon"].unique()
    items = train["work_id"].unique()

    user_to_index = {user: index for index, user in enumerate(users)}
    item_to_index = {item: index for index, item in enumerate(items)}

    n_users = len(users)
    n_items = len(items)
    all_items = np.arange(n_items, dtype=np.int64)

    train_users = train["customer_id_anon"].map(user_to_index).to_numpy(
        dtype=np.int64
    )
    train_items = train["work_id"].map(item_to_index).to_numpy(
        dtype=np.int64
    )

    positives_by_user = defaultdict(list)
    for user_index, item_index in zip(train_users, train_items):
        positives_by_user[int(user_index)].append(int(item_index))

    user_positive_items = [
        np.array(positives_by_user.get(user, []), dtype=np.int64)
        for user in range(n_users)
    ]

    test_dict = {}
    for raw_user, raw_item in zip(
        test["customer_id_anon"],
        test["work_id"],
    ):
        if raw_user in user_to_index and raw_item in item_to_index:
            test_dict[user_to_index[raw_user]] = item_to_index[raw_item]

    rng = np.random.default_rng(SEED_NEG)
    negative_user_blocks = []
    negative_item_blocks = []

    for user in range(n_users):
        positive_items = user_positive_items[user]

        if len(positive_items) == 0:
            continue

        excluded_items = positive_items

        if (
            EXCLUDE_TEST_ITEM_FROM_TRAIN_NEGATIVES
            and user in test_dict
        ):
            excluded_items = np.append(
                excluded_items,
                test_dict[user],
            )

        pool = np.setdiff1d(
            all_items,
            excluded_items,
            assume_unique=False,
        )

        number_of_negatives = min(
            neg_ratio * len(positive_items),
            len(pool),
        )

        sampled_negatives = rng.choice(
            pool,
            size=number_of_negatives,
            replace=False,
        )

        negative_user_blocks.append(
            np.full(number_of_negatives, user, dtype=np.int64)
        )
        negative_item_blocks.append(
            sampled_negatives.astype(np.int64)
        )

    negative_users = np.concatenate(negative_user_blocks)
    negative_items = np.concatenate(negative_item_blocks)

    omega_users = np.concatenate(
        [train_users, negative_users]
    ).astype(np.int64)
    omega_items = np.concatenate(
        [train_items, negative_items]
    ).astype(np.int64)
    omega_labels = np.concatenate(
        [
            np.ones(len(train_users), dtype=np.float32),
            np.zeros(len(negative_users), dtype=np.float32),
        ]
    )

    if EXCLUDE_TEST_ITEM_FROM_TRAIN_NEGATIVES:
        negative_pairs = set(
            zip(negative_users.tolist(), negative_items.tolist())
        )
        contradictory_test_labels = sum(
            (user, item) in negative_pairs
            for user, item in test_dict.items()
        )

        if contradictory_test_labels != 0:
            raise AssertionError(
                f"MIN{min_interactions}: {contradictory_test_labels} "
                "held-out positives were sampled as training negatives."
            )

    omega_size = len(omega_users)
    nominal_k_star = omega_size / (n_users + n_items)

    report = {
        "dataset": f"MIN{min_interactions}",
        "min_interactions": min_interactions,
        "users": n_users,
        "items": n_items,
        "train_positives": len(train_users),
        "train_negatives": len(negative_users),
        "omega_size": omega_size,
        "test_users": len(test_dict),
        "cold_test_items_removed": cold_removed,
        "kept_interactions_before_split": kept_interactions,
        "negative_ratio_train": neg_ratio,
        "nominal_k_star": nominal_k_star,
        "test_positive_excluded_from_train_negatives": (
            EXCLUDE_TEST_ITEM_FROM_TRAIN_NEGATIVES
        ),
        "train_path": str(train_path),
        "test_path": str(test_path),
    }

    return {
        "train": train,
        "test": test,
        "n_users": n_users,
        "n_items": n_items,
        "all_items": all_items,
        "train_users": train_users,
        "train_items": train_items,
        "user_positive_items": user_positive_items,
        "test_dict": test_dict,
        "omega": (omega_users, omega_items, omega_labels),
        "report": report,
    }


def build_eval_pointwise(data, n_negatives, seed=SEED_EVAL):
    rng = np.random.default_rng(seed)

    eval_users = []
    eval_items = []
    eval_labels = []

    for user, held_out_item in data["test_dict"].items():
        eval_users.append(user)
        eval_items.append(held_out_item)
        eval_labels.append(1.0)

        excluded_items = np.append(
            data["user_positive_items"][user],
            held_out_item,
        )

        pool = np.setdiff1d(
            data["all_items"],
            excluded_items,
            assume_unique=False,
        )

        sampled_negatives = rng.choice(
            pool,
            size=min(n_negatives, len(pool)),
            replace=False,
        )

        eval_users.extend([user] * len(sampled_negatives))
        eval_items.extend(sampled_negatives.tolist())
        eval_labels.extend([0.0] * len(sampled_negatives))

    return (
        np.asarray(eval_users, dtype=np.int64),
        np.asarray(eval_items, dtype=np.int64),
        np.asarray(eval_labels, dtype=np.float32),
    )

## 6. Logistic MF Model

The score combines a global intercept, user and item biases, and the inner product
of the latent vectors. Binary cross-entropy is applied to the resulting logits.

In [ ]:
class LogisticMF(nn.Module):
    def __init__(self, n_users, n_items, latent_dimension):
        super().__init__()

        self.user_factors = nn.Embedding(
            n_users,
            latent_dimension,
        )
        self.item_factors = nn.Embedding(
            n_items,
            latent_dimension,
        )
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(
            self.user_factors.weight,
            mean=0.0,
            std=0.01,
        )
        nn.init.normal_(
            self.item_factors.weight,
            mean=0.0,
            std=0.01,
        )
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_indices, item_indices):
        user_vectors = self.user_factors(user_indices)
        item_vectors = self.item_factors(item_indices)

        interaction = (
            user_vectors * item_vectors
        ).sum(dim=1)

        user_effect = self.user_bias(
            user_indices
        ).squeeze(1)
        item_effect = self.item_bias(
            item_indices
        ).squeeze(1)

        return (
            self.global_bias
            + user_effect
            + item_effect
            + interaction
        )


def logistic_mf_parameter_count(
    latent_dimension,
    n_users,
    n_items,
):
    return (
        (n_users + n_items) * latent_dimension
        + n_users
        + n_items
        + 1
    )

## 7. Training and Evaluation Functions

The functions below train independent deterministic models, compute train and
test RMSE, evaluate full-catalog Recall and NDCG, save optimization diagnostics,
and checkpoint each completed configuration atomically.

In [ ]:
def choose_adaptive_batch_size(latent_dimension):
    if latent_dimension >= 3072:
        return 512
    if latent_dimension >= 1536:
        return 768
    if latent_dimension >= 768:
        return 1024
    if latent_dimension >= 384:
        return 2048
    return 4096


def predict_probabilities(
    model,
    users,
    items,
    batch_size=262_144,
):
    model.eval()
    probabilities = []

    with torch.no_grad():
        for start in range(0, len(users), batch_size):
            end = min(start + batch_size, len(users))

            user_tensor = torch.as_tensor(
                users[start:end],
                dtype=torch.long,
                device=device,
            )
            item_tensor = torch.as_tensor(
                items[start:end],
                dtype=torch.long,
                device=device,
            )

            logits = model(user_tensor, item_tensor)
            probabilities.append(
                torch.sigmoid(logits).cpu().numpy()
            )

    return np.concatenate(probabilities)


def root_mean_squared_error(labels, probabilities):
    return float(
        np.sqrt(
            np.mean(
                (labels - probabilities) ** 2
            )
        )
    )


def rmse_on_pairs(model, pair_data):
    users, items, labels = pair_data
    probabilities = predict_probabilities(
        model,
        users,
        items,
    )
    return root_mean_squared_error(
        labels,
        probabilities,
    )


def ranking_metrics(
    model,
    data,
    cutoffs=(5, 10, 20),
    user_batch_size=256,
):
    model.eval()

    test_users = np.fromiter(
        data["test_dict"].keys(),
        dtype=np.int64,
    )
    held_out_items = np.fromiter(
        (
            data["test_dict"][user]
            for user in test_users
        ),
        dtype=np.int64,
    )

    ranks = []

    with torch.no_grad():
        item_factors = model.item_factors.weight
        item_bias = model.item_bias.weight.squeeze(1)

        for start in range(
            0,
            len(test_users),
            user_batch_size,
        ):
            end = min(
                start + user_batch_size,
                len(test_users),
            )

            batch_users_np = test_users[start:end]
            batch_held_out = held_out_items[start:end]

            batch_users = torch.as_tensor(
                batch_users_np,
                dtype=torch.long,
                device=device,
            )

            scores = (
                model.user_factors(batch_users)
                @ item_factors.T
            )
            scores = scores + item_bias.unsqueeze(0)
            scores = scores + model.user_bias(
                batch_users
            ).squeeze(1).unsqueeze(1)
            scores = scores + model.global_bias

            scores = scores.detach().cpu().numpy()

            for row, user in enumerate(batch_users_np):
                scores[
                    row,
                    data["user_positive_items"][user],
                ] = -np.inf

            held_out_scores = scores[
                np.arange(len(batch_users_np)),
                batch_held_out,
            ]

            batch_ranks = (
                (scores > held_out_scores[:, None])
                .sum(axis=1)
                + 1
            )

            ranks.append(batch_ranks)

    ranks = np.concatenate(ranks)

    metrics = {}

    for cutoff in cutoffs:
        hits = ranks <= cutoff
        metrics[f"Recall@{cutoff}"] = float(
            hits.mean()
        )
        metrics[f"NDCG@{cutoff}"] = float(
            (
                hits
                / np.log2(ranks + 1)
            ).mean()
        )

    return metrics


def embedding_diagnostics(model):
    with torch.no_grad():
        return {
            "user_embedding_norm_mean": float(
                model.user_factors.weight.norm(
                    dim=1
                ).mean().item()
            ),
            "item_embedding_norm_mean": float(
                model.item_factors.weight.norm(
                    dim=1
                ).mean().item()
            ),
            "user_bias_abs_mean": float(
                model.user_bias.weight.abs().mean().item()
            ),
            "item_bias_abs_mean": float(
                model.item_bias.weight.abs().mean().item()
            ),
            "global_bias": float(
                model.global_bias.item()
            ),
        }


def train_logistic_mf(
    data,
    latent_dimension,
    n_epochs,
    learning_rate,
    weight_decay,
    batch_size,
    grad_clip_norm,
    seed=SEED,
):
    omega_users, omega_items, omega_labels = data["omega"]

    set_seed(seed)

    model = LogisticMF(
        data["n_users"],
        data["n_items"],
        latent_dimension,
    ).to(device)

    training_dataset = TensorDataset(
        torch.as_tensor(
            omega_users,
            dtype=torch.long,
        ),
        torch.as_tensor(
            omega_items,
            dtype=torch.long,
        ),
        torch.as_tensor(
            omega_labels,
            dtype=torch.float32,
        ),
    )

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)

    training_loader = DataLoader(
        training_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        generator=loader_generator,
    )

    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    checkpoint_epochs = sorted(
        {
            epoch
            for epoch in [
                1,
                5,
                10,
                20,
                30,
                50,
                100,
                200,
                n_epochs,
            ]
            if epoch <= n_epochs
        }
    )

    loss_history = {}

    for epoch in range(1, n_epochs + 1):
        model.train()

        total_loss = 0.0
        total_examples = 0

        for (
            batch_users,
            batch_items,
            batch_labels,
        ) in training_loader:
            batch_users = batch_users.to(
                device,
                non_blocking=True,
            )
            batch_items = batch_items.to(
                device,
                non_blocking=True,
            )
            batch_labels = batch_labels.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(set_to_none=True)

            logits = model(
                batch_users,
                batch_items,
            )
            loss = loss_function(
                logits,
                batch_labels,
            )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    "Non-finite loss at "
                    f"epoch={epoch}, k={latent_dimension}."
                )

            loss.backward()

            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=grad_clip_norm,
                )

            optimizer.step()

            number_in_batch = len(batch_labels)
            total_loss += (
                loss.item() * number_in_batch
            )
            total_examples += number_in_batch

        epoch_loss = total_loss / total_examples

        if not np.isfinite(epoch_loss):
            raise FloatingPointError(
                "Non-finite epoch loss at "
                f"epoch={epoch}, k={latent_dimension}."
            )

        if epoch in checkpoint_epochs:
            loss_history[epoch] = float(epoch_loss)

    return (
        model,
        float(epoch_loss),
        loss_history,
    )


def evaluate_model(
    model,
    data,
    evaluation_100,
    evaluation_4,
):
    return {
        "rmse_train": rmse_on_pairs(
            model,
            data["omega"],
        ),
        "rmse_test_100": rmse_on_pairs(
            model,
            evaluation_100,
        ),
        "rmse_test_4": rmse_on_pairs(
            model,
            evaluation_4,
        ),
        **ranking_metrics(model, data),
    }


def atomic_save_csv(dataframe, path):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    dataframe.to_csv(
        temporary_path,
        index=False,
    )

    with open(temporary_path, "rb") as file:
        os.fsync(file.fileno())

    os.replace(
        temporary_path,
        path,
    )


def save_result_rows(rows, path):
    if len(rows) == 0:
        result = pd.DataFrame()
    else:
        result = (
            pd.DataFrame(rows)
            .drop_duplicates(
                subset=[
                    "min_interactions",
                    "k",
                    "epochs",
                ],
                keep="last",
            )
            .sort_values(
                [
                    "min_interactions",
                    "k",
                    "epochs",
                ]
            )
            .reset_index(drop=True)
        )

    atomic_save_csv(result, path)
    return result


def run_experiment_grid(
    *,
    experiment_name,
    result_path,
    min_grid,
    k_grid,
    epoch_grid,
    learning_rate,
    weight_decay,
    fixed_batch_size,
    grad_clip_norm,
    regularization_label,
):
    if (
        RESUME_FROM_WORKING
        and Path(result_path).exists()
    ):
        existing_results = pd.read_csv(
            result_path
        )
        rows = existing_results.to_dict(
            "records"
        )
        print(
            "Loaded local checkpoint:",
            result_path,
            existing_results.shape,
        )
    else:
        rows = []
        print("Starting from scratch:", result_path)

    completed = {
        (
            int(row["min_interactions"]),
            int(row["k"]),
            int(row["epochs"]),
        )
        for row in rows
        if row.get("status", "completed") == "completed"
    }

    print("Completed configurations:", len(completed))

    try:
        for min_interactions in min_grid:
            data = prepared[min_interactions]
            evaluation_100, evaluation_4 = eval_cache[
                min_interactions
            ]

            for latent_dimension in k_grid:
                for n_epochs in epoch_grid:
                    key = (
                        min_interactions,
                        latent_dimension,
                        n_epochs,
                    )

                    if key in completed:
                        print("Skipping completed:", key)
                        continue

                    batch_size = (
                        fixed_batch_size
                        if fixed_batch_size is not None
                        else choose_adaptive_batch_size(
                            latent_dimension
                        )
                    )

                    number_of_parameters = (
                        logistic_mf_parameter_count(
                            latent_dimension,
                            data["n_users"],
                            data["n_items"],
                        )
                    )

                    parameters_over_omega = (
                        number_of_parameters
                        / data["report"]["omega_size"]
                    )

                    print("\n" + "=" * 100)
                    print(
                        f"{experiment_name} | "
                        f"MIN{min_interactions} | "
                        f"k={latent_dimension} | "
                        f"epochs={n_epochs} | "
                        f"lr={learning_rate} | "
                        f"batch={batch_size} | "
                        f"grad_clip={grad_clip_norm} | "
                        f"seed={SEED}"
                    )
                    print("=" * 100)

                    start_time = time.time()
                    model = None

                    try:
                        (
                            model,
                            train_bce,
                            loss_history,
                        ) = train_logistic_mf(
                            data=data,
                            latent_dimension=latent_dimension,
                            n_epochs=n_epochs,
                            learning_rate=learning_rate,
                            weight_decay=weight_decay,
                            batch_size=batch_size,
                            grad_clip_norm=grad_clip_norm,
                            seed=SEED,
                        )

                        metrics = evaluate_model(
                            model,
                            data,
                            evaluation_100,
                            evaluation_4,
                        )

                        diagnostics = embedding_diagnostics(
                            model
                        )

                        record = {
                            "experiment": experiment_name,
                            "dataset": f"MIN{min_interactions}",
                            "min_interactions": min_interactions,
                            "model": "LogisticMF",
                            "regularization_label": regularization_label,
                            "weight_decay": weight_decay,
                            "learning_rate": learning_rate,
                            "negative_ratio_train": NEG_RATIO_TRAIN,
                            "k": latent_dimension,
                            "epochs": n_epochs,
                            "batch_size": batch_size,
                            "grad_clip_norm": (
                                np.nan
                                if grad_clip_norm is None
                                else grad_clip_norm
                            ),
                            "training_seed": SEED,
                            "negative_sampling_seed": SEED_NEG,
                            "evaluation_seed": SEED_EVAL,
                            "num_parameters": number_of_parameters,
                            "parameters_over_omega": parameters_over_omega,
                            "nominal_k_star": data["report"][
                                "nominal_k_star"
                            ],
                            "omega_size": data["report"][
                                "omega_size"
                            ],
                            "train_bce": train_bce,
                            "loss_history_json": json.dumps(
                                loss_history
                            ),
                            **metrics,
                            **diagnostics,
                            "sec": round(
                                time.time() - start_time,
                                1,
                            ),
                            "status": "completed",
                            "error": "",
                        }

                        rows.append(record)
                        completed.add(key)

                        current_results = save_result_rows(
                            rows,
                            result_path,
                        )

                        print(
                            f"Checkpoint saved: "
                            f"{current_results.shape[0]} rows"
                        )
                        print(record)

                    except Exception as error:
                        failure_record = {
                            "experiment": experiment_name,
                            "dataset": f"MIN{min_interactions}",
                            "min_interactions": min_interactions,
                            "model": "LogisticMF",
                            "regularization_label": regularization_label,
                            "weight_decay": weight_decay,
                            "learning_rate": learning_rate,
                            "negative_ratio_train": NEG_RATIO_TRAIN,
                            "k": latent_dimension,
                            "epochs": n_epochs,
                            "batch_size": batch_size,
                            "grad_clip_norm": (
                                np.nan
                                if grad_clip_norm is None
                                else grad_clip_norm
                            ),
                            "training_seed": SEED,
                            "negative_sampling_seed": SEED_NEG,
                            "evaluation_seed": SEED_EVAL,
                            "num_parameters": number_of_parameters,
                            "parameters_over_omega": parameters_over_omega,
                            "nominal_k_star": data["report"][
                                "nominal_k_star"
                            ],
                            "omega_size": data["report"][
                                "omega_size"
                            ],
                            "train_bce": np.nan,
                            "loss_history_json": "",
                            "rmse_train": np.nan,
                            "rmse_test_100": np.nan,
                            "rmse_test_4": np.nan,
                            "Recall@5": np.nan,
                            "NDCG@5": np.nan,
                            "Recall@10": np.nan,
                            "NDCG@10": np.nan,
                            "Recall@20": np.nan,
                            "NDCG@20": np.nan,
                            "user_embedding_norm_mean": np.nan,
                            "item_embedding_norm_mean": np.nan,
                            "user_bias_abs_mean": np.nan,
                            "item_bias_abs_mean": np.nan,
                            "global_bias": np.nan,
                            "sec": round(
                                time.time() - start_time,
                                1,
                            ),
                            "status": "failed",
                            "error": str(error),
                        }

                        rows.append(failure_record)
                        save_result_rows(
                            rows,
                            result_path,
                        )

                        print(
                            "FAILED:",
                            key,
                            repr(error),
                        )

                    finally:
                        if model is not None:
                            del model

                        gc.collect()
                        torch.cuda.empty_cache()

    except KeyboardInterrupt:
        save_result_rows(rows, result_path)
        print(
            "\nManual interruption detected. "
            "All completed configurations are saved."
        )
        raise

    final_results = save_result_rows(
        rows,
        result_path,
    )

    print("Final checkpoint:", result_path)
    print("Final shape:", final_results.shape)

    return final_results

## 8. Prepare MIN10, MIN20, and MIN25

The filtered datasets, fixed splits, fixed sampled supports, and fixed evaluation
sets are constructed once and reused by every experiment.

In [ ]:
prepared = {}
eval_cache = {}
protocol_reports = []

for min_interactions in MIN_INTERACTIONS_GRID:
    print("\nPreparing", f"MIN{min_interactions}")

    prepared[min_interactions] = prepare_protocol_data(
        df,
        min_interactions=min_interactions,
        neg_ratio=NEG_RATIO_TRAIN,
    )

    evaluation_100 = build_eval_pointwise(
        prepared[min_interactions],
        n_negatives=100,
        seed=SEED_EVAL,
    )
    evaluation_4 = build_eval_pointwise(
        prepared[min_interactions],
        n_negatives=4,
        seed=SEED_EVAL,
    )

    eval_cache[min_interactions] = (
        evaluation_100,
        evaluation_4,
    )

    protocol_reports.append(
        prepared[min_interactions]["report"]
    )

protocol_summary = pd.DataFrame(
    protocol_reports
).sort_values("min_interactions")

protocol_summary_path = (
    SUMMARY_DIR / "protocol_summary.csv"
)
protocol_summary.to_csv(
    protocol_summary_path,
    index=False,
)

display(protocol_summary)
print("Saved:", protocol_summary_path)

## 9. Initial 30-Epoch Capacity Sweep

This is the original full-capacity experiment: \(lr=0.01\), 30 epochs, weak
weight decay, and adaptive batch size across MIN10, MIN20, and MIN25.

In [ ]:
if RUN_INITIAL_SWEEP:
    initial_results = run_experiment_grid(
        experiment_name="initial_30_epoch_capacity_sweep",
        result_path=INITIAL_RESULTS_PATH,
        min_grid=MIN_INTERACTIONS_GRID,
        k_grid=INITIAL_K_GRID,
        epoch_grid=INITIAL_EPOCHS,
        learning_rate=INITIAL_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        fixed_batch_size=None,
        grad_clip_norm=INITIAL_GRAD_CLIP,
        regularization_label="wd_1e-6_lr_0.01_initial",
    )

    display(
        initial_results[
            initial_results["status"] == "completed"
        ].sort_values(
            ["min_interactions", "k"]
        )
    )
else:
    print("Initial sweep disabled by configuration.")

## 10. Training-Budget Diagnostic: \(lr=0.01\)

This controlled MIN10 experiment compares 30, 50, 100, and 200 epochs using a
fixed batch size of 1024 and the original learning rate.

In [ ]:
if RUN_TRAINING_BUDGET_DIAGNOSTIC:
    training_budget_results = run_experiment_grid(
        experiment_name="high_k_training_budget_diagnostic_lr001",
        result_path=TRAINING_BUDGET_RESULTS_PATH,
        min_grid=DIAGNOSTIC_MIN_GRID,
        k_grid=HIGH_K_GRID,
        epoch_grid=DIAGNOSTIC_EPOCH_GRID,
        learning_rate=TRAINING_BUDGET_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        fixed_batch_size=DIAGNOSTIC_BATCH_SIZE,
        grad_clip_norm=TRAINING_BUDGET_GRAD_CLIP,
        regularization_label="wd_1e-6_lr_0.01_high_k_diagnostic",
    )

    display(
        training_budget_results[
            training_budget_results["status"] == "completed"
        ].sort_values(
            ["epochs", "k"]
        )
    )
else:
    print("Training-budget diagnostic disabled by configuration.")

## 11. Learning-Rate Stabilization Diagnostic: \(lr=0.003\)

The high-\(k\) grid, epochs, batch size, data, and seeds are unchanged from the
previous diagnostic. Only the learning rate is reduced, isolating its effect on
optimization stability.

In [ ]:
if RUN_STABILITY_DIAGNOSTIC:
    stability_results = run_experiment_grid(
        experiment_name="high_k_learning_rate_stability_lr003",
        result_path=STABILITY_RESULTS_PATH,
        min_grid=DIAGNOSTIC_MIN_GRID,
        k_grid=HIGH_K_GRID,
        epoch_grid=DIAGNOSTIC_EPOCH_GRID,
        learning_rate=STABILITY_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        fixed_batch_size=DIAGNOSTIC_BATCH_SIZE,
        grad_clip_norm=STABILITY_DIAGNOSTIC_GRAD_CLIP,
        regularization_label="wd_1e-6_lr_0.003_high_k_diagnostic",
    )

    display(
        stability_results[
            stability_results["status"] == "completed"
        ].sort_values(
            ["epochs", "k"]
        )
    )
else:
    print("Learning-rate diagnostic disabled by configuration.")

## 12. Final Stabilized Full-Capacity Sweep

The final reported sweep uses \(lr=0.003\), 50 epochs, batch size 1024, weight
decay \(10^{-6}\), and gradient clipping with norm 1.0 across the full MIN10 and
MIN20 capacity ranges.

In [ ]:
if RUN_FINAL_STABLE_SWEEP:
    final_stable_results = run_experiment_grid(
        experiment_name="final_stabilized_full_capacity_sweep",
        result_path=FINAL_STABLE_RESULTS_PATH,
        min_grid=FINAL_STABLE_MIN_GRID,
        k_grid=FINAL_STABLE_K_GRID,
        epoch_grid=FINAL_STABLE_EPOCHS,
        learning_rate=FINAL_STABLE_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        fixed_batch_size=FINAL_STABLE_BATCH_SIZE,
        grad_clip_norm=FINAL_STABLE_GRAD_CLIP,
        regularization_label=(
            "wd_1e-6_lr_0.003_epochs_50_batch_1024_gradclip_1.0"
        ),
    )

    display(
        final_stable_results[
            final_stable_results["status"] == "completed"
        ].sort_values(
            ["min_interactions", "k"]
        )
    )
else:
    print("Final stabilized sweep disabled by configuration.")

## 13. Final Figures

All figures are generated directly from the CSV files created by the preceding
experiment cells. No values are hard-coded or reconstructed from external files.
PNG and PDF versions are saved.

In [ ]:
def load_completed_results(path):
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Required result file not found: {path}"
        )

    result = pd.read_csv(path)

    if "status" in result.columns:
        result = result[
            result["status"] == "completed"
        ].copy()

    if result.empty:
        raise ValueError(
            f"No completed rows were found in {path}."
        )

    return result


def save_current_figure(filename_stem):
    png_path = FIGURES_DIR / f"{filename_stem}.png"
    pdf_path = FIGURES_DIR / f"{filename_stem}.pdf"

    plt.tight_layout()
    plt.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.savefig(
        pdf_path,
        bbox_inches="tight",
    )
    plt.show()
    plt.close()

    print("Saved:", png_path)
    print("Saved:", pdf_path)


def plot_metric_by_complexity(
    result,
    metric,
    ylabel,
    title,
    filename_stem,
):
    plt.figure(figsize=(8, 5))

    for min_interactions, group in result.groupby(
        "min_interactions"
    ):
        ordered = group.sort_values(
            "parameters_over_omega"
        )

        plt.plot(
            ordered["parameters_over_omega"],
            ordered[metric],
            marker="o",
            linewidth=1.8,
            label=f"MIN{int(min_interactions)}",
        )

    plt.axvline(
        1.0,
        linestyle="--",
        linewidth=1.2,
        label=r"parameters / $|\Omega|$ = 1",
    )
    plt.xscale("log")
    plt.xlabel(
        "Model complexity: parameters / sampled training observations"
    )
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(
        True,
        which="both",
        alpha=0.3,
    )
    plt.legend()

    save_current_figure(filename_stem)


def plot_high_k_epochs(
    result,
    metric,
    ylabel,
    title,
    filename_stem,
    y_limits=None,
):
    plt.figure(figsize=(8, 5))

    for n_epochs, group in result.groupby("epochs"):
        ordered = group.sort_values("k")

        plt.plot(
            ordered["k"],
            ordered[metric],
            marker="o",
            linewidth=1.8,
            label=f"{int(n_epochs)} epochs",
        )

    nominal_k_star = float(
        result["nominal_k_star"].iloc[0]
    )

    plt.axvline(
        nominal_k_star,
        linestyle="--",
        linewidth=1.2,
        label=rf"Nominal threshold $k^\star\approx{nominal_k_star:.1f}$",
    )

    plt.xscale("log", base=2)
    plt.xticks(
        HIGH_K_GRID,
        [str(value) for value in HIGH_K_GRID],
    )
    plt.xlabel("Latent dimension k")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(
        True,
        which="both",
        alpha=0.3,
    )
    plt.legend(title="Training budget")

    if y_limits is not None:
        plt.ylim(*y_limits)

    save_current_figure(filename_stem)


initial_plot_data = load_completed_results(
    INITIAL_RESULTS_PATH
)
training_budget_plot_data = load_completed_results(
    TRAINING_BUDGET_RESULTS_PATH
)
stability_plot_data = load_completed_results(
    STABILITY_RESULTS_PATH
)
final_stable_plot_data = load_completed_results(
    FINAL_STABLE_RESULTS_PATH
)

# Shared scales make the before/after diagnostic comparisons direct.
combined_test_values = pd.concat(
    [
        training_budget_plot_data["rmse_test_100"],
        stability_plot_data["rmse_test_100"],
    ]
).dropna()

test_range = (
    float(combined_test_values.max())
    - float(combined_test_values.min())
)
test_padding = max(0.01, 0.05 * test_range)
shared_test_limits = (
    max(
        0.0,
        float(combined_test_values.min()) - test_padding,
    ),
    float(combined_test_values.max()) + test_padding,
)

combined_train_values = pd.concat(
    [
        training_budget_plot_data["rmse_train"],
        stability_plot_data["rmse_train"],
    ]
).dropna()

shared_train_limits = (
    0.0,
    float(combined_train_values.max()) * 1.05,
)

plot_metric_by_complexity(
    initial_plot_data,
    metric="rmse_test_100",
    ylabel="Test RMSE, 1:100 negatives",
    title="Initial Logistic MF: test RMSE across model complexity",
    filename_stem="original_rmse_test_100_vs_complexity",
)

plot_metric_by_complexity(
    initial_plot_data,
    metric="rmse_train",
    ylabel="Train RMSE",
    title="Initial Logistic MF: training RMSE across model complexity",
    filename_stem="original_train_rmse_vs_complexity",
)

plot_high_k_epochs(
    training_budget_plot_data,
    metric="rmse_test_100",
    ylabel="Test RMSE, 1:100 negatives",
    title="High-k optimization sensitivity: MIN10, lr=0.01",
    filename_stem=(
        "fig_highk_non_stabilized_lr001_epochs_30_50_100_200"
    ),
    y_limits=shared_test_limits,
)

plot_high_k_epochs(
    training_budget_plot_data,
    metric="rmse_train",
    ylabel="Train RMSE",
    title="High-k training instability: MIN10, lr=0.01",
    filename_stem=(
        "fig_highk_non_stabilized_train_rmse_lr001"
    ),
    y_limits=shared_train_limits,
)

plot_high_k_epochs(
    stability_plot_data,
    metric="rmse_test_100",
    ylabel="Test RMSE, 1:100 negatives",
    title="Stabilized high-k scan: MIN10, lr=0.003",
    filename_stem=(
        "fig_highk_stabilized_lr003_epochs_30_50_100_200"
    ),
    y_limits=shared_test_limits,
)

plot_high_k_epochs(
    stability_plot_data,
    metric="rmse_train",
    ylabel="Train RMSE",
    title="Stabilized high-k training error: MIN10, lr=0.003",
    filename_stem=(
        "fig_highk_stabilized_train_rmse_lr003"
    ),
    y_limits=shared_train_limits,
)

plot_metric_by_complexity(
    final_stable_plot_data,
    metric="rmse_test_100",
    ylabel="Test RMSE, 1:100 negatives",
    title="Stabilized Logistic MF: test RMSE across model complexity",
    filename_stem="stable_rmse_test_100_vs_complexity",
)

plot_metric_by_complexity(
    final_stable_plot_data,
    metric="rmse_train",
    ylabel="Train RMSE",
    title="Stabilized Logistic MF: training RMSE across model complexity",
    filename_stem="stable_rmse_train_vs_complexity",
)

plt.figure(figsize=(8, 5))
for min_interactions, group in final_stable_plot_data.groupby(
    "min_interactions"
):
    ordered = group.sort_values("k")
    plt.plot(
        ordered["k"],
        ordered["NDCG@10"],
        marker="o",
        linewidth=1.8,
        label=f"MIN{int(min_interactions)}",
    )
plt.xscale("log", base=2)
plt.xlabel("Latent dimension k")
plt.ylabel("NDCG@10")
plt.title("Stabilized Logistic MF: NDCG@10 across latent dimension")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
save_current_figure("stable_ndcg10_vs_k")

plt.figure(figsize=(8, 5))
for min_interactions, group in final_stable_plot_data.groupby(
    "min_interactions"
):
    ordered = group.sort_values("k")
    plt.plot(
        ordered["k"],
        ordered["Recall@10"],
        marker="o",
        linewidth=1.8,
        label=f"MIN{int(min_interactions)}",
    )
plt.xscale("log", base=2)
plt.xlabel("Latent dimension k")
plt.ylabel("Recall@10")
plt.title("Stabilized Logistic MF: Recall@10 across latent dimension")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
save_current_figure("stable_recall10_vs_k")

## 14. Summary Tables

This section reports experiment completeness, best final-sweep configurations,
and the direct \(k=2048\) comparison across learning rates and training budgets.

In [ ]:
result_specs = [
    {
        "experiment": "Initial 30-epoch sweep",
        "path": INITIAL_RESULTS_PATH,
        "expected_completed": (
            len(MIN_INTERACTIONS_GRID)
            * len(INITIAL_K_GRID)
            * len(INITIAL_EPOCHS)
        ),
    },
    {
        "experiment": "Training-budget diagnostic",
        "path": TRAINING_BUDGET_RESULTS_PATH,
        "expected_completed": (
            len(DIAGNOSTIC_MIN_GRID)
            * len(HIGH_K_GRID)
            * len(DIAGNOSTIC_EPOCH_GRID)
        ),
    },
    {
        "experiment": "Learning-rate diagnostic",
        "path": STABILITY_RESULTS_PATH,
        "expected_completed": (
            len(DIAGNOSTIC_MIN_GRID)
            * len(HIGH_K_GRID)
            * len(DIAGNOSTIC_EPOCH_GRID)
        ),
    },
    {
        "experiment": "Final stabilized sweep",
        "path": FINAL_STABLE_RESULTS_PATH,
        "expected_completed": (
            len(FINAL_STABLE_MIN_GRID)
            * len(FINAL_STABLE_K_GRID)
            * len(FINAL_STABLE_EPOCHS)
        ),
    },
]

completeness_rows = []

for specification in result_specs:
    result = pd.read_csv(specification["path"])

    completed_count = int(
        (
            result.get(
                "status",
                pd.Series(
                    ["completed"] * len(result)
                ),
            )
            == "completed"
        ).sum()
    )

    completeness_rows.append(
        {
            "experiment": specification["experiment"],
            "result_file": str(specification["path"]),
            "expected_completed": specification[
                "expected_completed"
            ],
            "actual_completed": completed_count,
            "complete": (
                completed_count
                == specification["expected_completed"]
            ),
        }
    )

experiment_completeness = pd.DataFrame(
    completeness_rows
)

completeness_path = (
    SUMMARY_DIR / "experiment_completeness.csv"
)
experiment_completeness.to_csv(
    completeness_path,
    index=False,
)

metrics_to_optimize = {
    "rmse_test_100": "min",
    "rmse_test_4": "min",
    "Recall@10": "max",
    "NDCG@10": "max",
    "Recall@20": "max",
    "NDCG@20": "max",
}

best_rows = []

for min_interactions, group in final_stable_plot_data.groupby(
    "min_interactions"
):
    summary_row = {
        "dataset": f"MIN{int(min_interactions)}"
    }

    for metric, direction in metrics_to_optimize.items():
        if direction == "min":
            best_index = group[metric].idxmin()
        else:
            best_index = group[metric].idxmax()

        best_model = group.loc[best_index]

        summary_row[f"best_{metric}_k"] = int(
            best_model["k"]
        )
        summary_row[f"best_{metric}_value"] = float(
            best_model[metric]
        )
        summary_row[
            f"best_{metric}_parameters_over_omega"
        ] = float(
            best_model["parameters_over_omega"]
        )

    best_rows.append(summary_row)

best_final_metrics = pd.DataFrame(best_rows)
best_metrics_path = (
    SUMMARY_DIR / "best_final_stabilized_metrics.csv"
)
best_final_metrics.to_csv(
    best_metrics_path,
    index=False,
)

high_k_comparison = pd.concat(
    [
        training_budget_plot_data.assign(
            optimization_condition="lr=0.01"
        ),
        stability_plot_data.assign(
            optimization_condition="lr=0.003"
        ),
    ],
    ignore_index=True,
)

high_k_2048 = (
    high_k_comparison[
        high_k_comparison["k"] == 2048
    ][
        [
            "optimization_condition",
            "epochs",
            "learning_rate",
            "batch_size",
            "rmse_train",
            "rmse_test_100",
            "rmse_test_4",
            "Recall@10",
            "NDCG@10",
        ]
    ]
    .sort_values(
        [
            "optimization_condition",
            "epochs",
        ]
    )
    .reset_index(drop=True)
)

high_k_2048_path = (
    SUMMARY_DIR
    / "high_k_2048_learning_rate_epoch_comparison.csv"
)
high_k_2048.to_csv(
    high_k_2048_path,
    index=False,
)

print("\nEXPERIMENT COMPLETENESS")
display(experiment_completeness)

print("\nBEST FINAL STABILIZED METRICS")
display(best_final_metrics)

print("\nK=2048 DIAGNOSTIC COMPARISON")
display(high_k_2048)

print("Saved:", completeness_path)
print("Saved:", best_metrics_path)
print("Saved:", high_k_2048_path)

## 15. Export Results and Figures

A configuration file, file manifest, SHA-256 hashes, and a single downloadable ZIP
are created from the outputs generated by this notebook.

In [ ]:
experiment_configuration = {
    "input_file": str(INPUT_PATH),
    "only_external_input": "interactions_real_long.csv",
    "seeds": {
        "training": SEED,
        "negative_sampling": SEED_NEG,
        "evaluation": SEED_EVAL,
    },
    "negative_ratio_train": NEG_RATIO_TRAIN,
    "evaluation_negative_ratios": list(
        EVAL_NEGATIVE_RATIOS
    ),
    "weight_decay": WEIGHT_DECAY,
    "exclude_test_positive_from_training_negatives": (
        EXCLUDE_TEST_ITEM_FROM_TRAIN_NEGATIVES
    ),
    "experiments": {
        "initial_30_epoch_capacity_sweep": {
            "min_interactions": MIN_INTERACTIONS_GRID,
            "k_grid": INITIAL_K_GRID,
            "epochs": INITIAL_EPOCHS,
            "learning_rate": INITIAL_LEARNING_RATE,
            "batch_size": "adaptive",
            "gradient_clip": INITIAL_GRAD_CLIP,
        },
        "training_budget_diagnostic": {
            "min_interactions": DIAGNOSTIC_MIN_GRID,
            "k_grid": HIGH_K_GRID,
            "epochs": DIAGNOSTIC_EPOCH_GRID,
            "learning_rate": TRAINING_BUDGET_LEARNING_RATE,
            "batch_size": DIAGNOSTIC_BATCH_SIZE,
            "gradient_clip": TRAINING_BUDGET_GRAD_CLIP,
        },
        "learning_rate_stability_diagnostic": {
            "min_interactions": DIAGNOSTIC_MIN_GRID,
            "k_grid": HIGH_K_GRID,
            "epochs": DIAGNOSTIC_EPOCH_GRID,
            "learning_rate": STABILITY_LEARNING_RATE,
            "batch_size": DIAGNOSTIC_BATCH_SIZE,
            "gradient_clip": STABILITY_DIAGNOSTIC_GRAD_CLIP,
        },
        "final_stabilized_full_capacity_sweep": {
            "min_interactions": FINAL_STABLE_MIN_GRID,
            "k_grid": FINAL_STABLE_K_GRID,
            "epochs": FINAL_STABLE_EPOCHS,
            "learning_rate": FINAL_STABLE_LEARNING_RATE,
            "batch_size": FINAL_STABLE_BATCH_SIZE,
            "gradient_clip": FINAL_STABLE_GRAD_CLIP,
        },
    },
}

configuration_path = (
    METADATA_DIR / "experiment_configuration.json"
)

with open(
    configuration_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        experiment_configuration,
        file,
        indent=2,
    )


def sha256_file(path, chunk_size=1_048_576):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


manifest_rows = []

for path in sorted(
    file_path
    for file_path in OUTPUT_DIR.rglob("*")
    if file_path.is_file()
):
    manifest_rows.append(
        {
            "relative_path": str(
                path.relative_to(OUTPUT_DIR)
            ),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
    )

manifest = pd.DataFrame(manifest_rows)
manifest_path = METADATA_DIR / "file_manifest.csv"
manifest.to_csv(manifest_path, index=False)

ZIP_PATH = (
    KAGGLE_WORKING_DIR
    / "logistic_mf_final_results.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(
        file_path
        for file_path in OUTPUT_DIR.rglob("*")
        if file_path.is_file()
    ):
        archive.write(
            path,
            arcname=str(
                path.relative_to(OUTPUT_DIR)
            ),
        )

print("Configuration:", configuration_path)
print("Manifest:", manifest_path)
print("ZIP:", ZIP_PATH)
print("Files included:", len(manifest))

display(FileLink(str(ZIP_PATH)))